# データ読込

In [12]:
# ローカルユーティリティ: 外部ディレクトリ(utils, logic)への依存を排除
import pandas as pd
import sqlite3
from datetime import datetime, date, timedelta
from typing import Union, List
import jpholiday

# 日本の祝日取得（jpholiday が無ければ空集合）
def get_japanese_holidays(
    start: Union[str, date], end: Union[str, date], as_str: bool = True
) -> Union[List[str], List[date]]:
    """
    指定した期間の日本の祝日を取得する関数。

    Args:
        start (str or date): 開始日（"YYYY-MM-DD" または date型）
        end (str or date): 終了日（"YYYY-MM-DD" または date型）
        as_str (bool): Trueなら"YYYY-MM-DD"形式、Falseならdate型

    Returns:
        Union[List[str], List[date]]: 祝日のリスト（文字列またはdate型）
    """
    # --- 日付型でなければ変換 ---
    if isinstance(start, str):
        start = datetime.strptime(start, "%Y-%m-%d").date()
    if isinstance(end, str):
        end = datetime.strptime(end, "%Y-%m-%d").date()

    # --- 日付範囲の祝日抽出 ---
    holidays = [
        d
        for d in (start + timedelta(days=i) for i in range((end - start).days + 1))
        if jpholiday.is_holiday(d)
    ]

    return [d.strftime("%Y-%m-%d") for d in holidays] if as_str else holidays

# 日本語フォント設定（存在する最初の候補を適用）
def set_jp_font():
    try:
        import matplotlib.pyplot as plt
        from matplotlib import font_manager
        candidates = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "TakaoGothic"]
        system_fonts = font_manager.findSystemFonts()
        for cand in candidates:
            for f in system_fonts:
                if cand in f:
                    plt.rcParams["font.family"] = cand
                    return
    except Exception:
        pass  # フォント設定失敗は無視

# SQLite から重量データを読む（テーブル名は推測。存在するテーブルに合わせて変更可）
def load_data_from_sqlite(db_path="/work/app/data/factory_manage/weight_data.db", table_name="weight_data"):
    with sqlite3.connect(db_path) as conn:
        # テーブル存在チェック & 自動選択
        try:
            df_tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
            if table_name not in df_tables['name'].tolist() and len(df_tables):
                # 最初のテーブルを利用
                table_name_local = df_tables['name'].iloc[0]
            else:
                table_name_local = table_name
        except Exception:
            table_name_local = table_name
        df_local = pd.read_sql(f"SELECT * FROM {table_name_local}", conn)
    # 日付カラム推測
    for col in ["伝票日付", "date", "dt"]:
        if col in df_local.columns:
            try:
                df_local[col] = pd.to_datetime(df_local[col])
            except Exception:
                pass
    return df_local

set_jp_font()
print("[INFO] ローカルユーティリティを読み込みました。")

# モジュールキャッシュのクリア（new_model1/new_model2 が部分的に読み込まれている場合に対応）
import sys, importlib
for name in list(sys.modules.keys()):
    if name.startswith('new_model1') or name.startswith('new_model2'):
        del sys.modules[name]
# 正しいパスを先頭に追加してからロード
sys.path.insert(0, '/home/ken/ai_work/works/scripts')
try:
    import new_model1
    importlib.reload(new_model1)
    print('reloaded new_model1 from', getattr(new_model1, '__file__', None))
except Exception as e:
    print('reload new_model1 failed:', e)
try:
    import new_model2
    importlib.reload(new_model2)
    print('reloaded new_model2 from', getattr(new_model2, '__file__', None))
except Exception as e:
    print('reload new_model2 failed:', e)


[INFO] ローカルユーティリティを読み込みました。
reload new_model1 failed: No module named 'new_model1'
reload new_model2 failed: No module named 'new_model2'


In [4]:
import pandas as pd
# from logic.factory_manage.utils.sql import load_data_from_sqlite  # 外部依存 -> ローカル版へ
# from utils.get_holydays import get_japanese_holidays             # 外部依存 -> ローカル版へ

# from utils.font import set_jp_font  # 外部フォント設定 -> ローカル版
# set_jp_font()  # 既に前セルで実行済み

# CSV / DB 読み込み（パスはPRE_HANNNYU配下に限定）
# path = "/works/data/factory_manage/weight_data.db"  # そのまま利用

# df = load_data_from_sqlite(path)
# df["伝票日付"].max()
# df.head()

### 2021年～

In [6]:
# pdは既にCELL INDEX:1でimportされているので、そのまま使えます
df_2021 = pd.read_csv("/works/data/input/2020顧客.csv", encoding="utf-8")
df_2022 = pd.read_csv("/works/data/input/2022顧客.csv", encoding="utf-8")
df_2023 = pd.read_csv("/works/data/input/2023_all.csv", encoding="utf-8")
df_2024 = pd.read_csv("/works/data/input/20240501-20250422.csv", encoding="utf-8")

df_2021 = df_2021[['伝票日付', '商品', '正味重量']]
df_2022 = df_2022[['伝票日付', '商品', '正味重量']]
df_2023 = df_2023[['伝票日付', '商品', '正味重量']]
df_2021.rename(columns={'商品': '品名'}, inplace=True)
df_2022.rename(columns={'商品': '品名'}, inplace=True)
df_2023.rename(columns={'商品': '品名'}, inplace=True)
df_2024 = df_2024[['伝票日付', '品名', '正味重量']]

df_all = pd.concat([df_2021, df_2022, df_2023, df_2024], ignore_index=True)
# 曜日など () を削除
df_all["伝票日付"] = df_all["伝票日付"].str.replace(r"\(.*\)", "", regex=True)
df_all["伝票日付"] = pd.to_datetime(df_all["伝票日付"], format="%Y/%m/%d")
df_all

/tmp/ipykernel_2888/3632036229.py:4: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2023 = pd.read_csv("/works/data/input/2023_all.csv", encoding="utf-8")


,伝票日付,品名,正味重量
0,2020-01-04,混合廃棄物A,870.0
1,2020-01-04,混合廃棄物A,450.0
2,2020-01-04,混合廃棄物（焼却物）,2400.0
3,2020-01-04,混合廃棄物A,250.0
4,2020-01-04,混合廃棄物A,800.0
...,...,...,...
184245,2025-05-26,軽量物系A(ｽﾀｲﾛﾌｫｰﾑ),10.0
184246,2025-05-26,廃ﾌﾟﾗｽﾁｯｸ類,30.0
184247,2025-05-26,廃ﾌﾟﾗｽﾁｯｸ類,160.0
184248,2025-05-26,混合廃棄物A,2530.0


### 予約情報

In [7]:
df_reserve = pd.read_csv("/work/data/input/yoyaku_data.csv")
df_reserve["予約日"] = pd.to_datetime(df_reserve["予約日"])
df_reserve.rename(columns={"台数": "予約台数"}, inplace=True)
print(df_reserve["予約日"].min(), df_reserve["予約日"].max())
print(df_reserve.columns)
df_reserve

2023-01-04 00:00:00 2025-05-31 00:00:00
Index(['予約日', '予約得意先名', '固定客', '予約台数'], dtype='object')


,予約日,予約得意先名,固定客,予約台数
0,2023-01-04,アンデス,False,1.0
1,2023-01-04,リサイクルレスキュー,False,1.0
2,2023-01-04,山口興業,False,2.0
3,2023-01-04,明和建装,False,1.0
4,2023-01-04,まごころ清掃社,False,1.0
...,...,...,...,...
45726,2025-05-31,首都高メンテナンス,False,1.0
45727,2025-05-31,シミズオクト,False,1.0
45728,2025-05-31,鈴亀,False,1.0
45729,2025-05-31,鈴木運輸,True,1.0


### 受入番号用

In [8]:
import os
import glob
import pandas as pd

# ディレクトリ内の全CSVファイルパスを取得
csv_dir = "/work/data/input/受入_時刻"
csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))

# 全CSVを読み込んで結合
dfs = []
for f in csv_files:
    df_tmp = pd.read_csv(f)

    # 伝票日付を整形 → 日付型へ変換
    df_tmp["伝票日付"] = df_tmp["伝票日付"].str.replace(r"\(.*?\)", "", regex=True).str.strip()
    df_tmp["伝票日付"] = pd.to_datetime(df_tmp["伝票日付"], format="%Y/%m/%d")

    # 正味重量をカンマ除去して数値化
    df_tmp["正味重量"] = df_tmp["正味重量"].replace({',': ''}, regex=True).astype(float)

    # 受入番号の欠損を埋めて型変換（念のため）
    df_tmp["受入番号"] = df_tmp["受入番号"].fillna(-1).astype(int)

    dfs.append(df_tmp)

df_Ukeire = pd.concat(dfs, ignore_index=True)

# 必要カラムだけ抽出（品名・重量・受入番号）
df_Ukeire = df_Ukeire[["伝票日付", "品名", "正味重量", "受入番号"]].copy()

# 台数カウント準備（日付・品名ごとの受入番号ユニーク数）
df_count = (
    df_Ukeire.groupby(["伝票日付", "品名"])["受入番号"]
    .nunique()
    .reset_index()
    .rename(columns={"受入番号": "台数"})
)

# （参考）結合しておく場合
df_merged = pd.merge(df_Ukeire, df_count, on=["伝票日付", "品名"], how="left")

# リネーム
df_merged.rename(columns={"台数": "搬入済台数"}, inplace=True)

# 結果確認
df_merged


,伝票日付,品名,正味重量,受入番号,搬入済台数
0,2024-12-01,混合廃棄物A,1100.0,31839,25
1,2024-12-01,運搬費,NaN,31839,1
2,2024-12-01,混合廃棄物A,720.0,31851,25
3,2024-12-01,混合廃棄物A,1860.0,31834,25
4,2024-12-01,混合廃棄物A,1060.0,31874,25
...,...,...,...,...,...
64680,2025-03-31,混合廃棄物A,170.0,46735,68
64681,2025-03-31,選別,80.0,46735,13
64682,2025-03-31,GC 軽鉄･ｽﾁｰﾙ類,160.0,46735,9
64683,2025-03-31,金属くず,400.0,46656,1


In [6]:
target_items = ["混合廃棄物A", "混合廃棄物B", "GC 軽鉄･ｽﾁｰﾙ類", "選別", "木くず"]

# 成功・モデル

## 予約数の追加モデル

In [7]:
# new_model1: PRE_HANNNYU/scripts/new_model1 内のローカルモジュールを動的パス追加で利用

import sys, os
SCRIPT_ROOT = "/works/scripts"
if SCRIPT_ROOT not in sys.path:
    sys.path.append(SCRIPT_ROOT)

# robust import

from new_model1 import full_walkforward, ReserveFeatureBuilder


import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt


In [8]:

# default: do not run full heavy pipeline unless explicitly enabled
RUN_PIPELINE = True

# 閾値（暫定的に緩和して予測生成が行われるか確認）
MIN_STAGE1_DAYS = 20  # 元:30
MIN_STAGE2_DAYS = 10  # 元:15

if RUN_PIPELINE:
    # 予約特徴量生成
    df_reserve_feat = ReserveFeatureBuilder(df_reserve).build()

    # 対象日を予約日に限定 (学習データが足りない場合はこのフィルタを緩めることを検討)
    reserve_dates = df_reserve_feat.index
    df_all["伝票日付"] = pd.to_datetime(df_all["伝票日付"])
    df_all = df_all[df_all["伝票日付"].isin(reserve_dates)].copy()

    print("[DEBUG] reservation_filtered_days=", df_all["伝票日付"].nunique())
    print("[DEBUG] reservation_span=", df_all["伝票日付"].min(), "->", df_all["伝票日付"].max())

    # 評価日数
    days_list = [300]
    results = []
    for days in days_list:
        print(f"\n=== {days}日分のデータで評価中 ===")
        latest_date = df_all["伝票日付"].max()
        cutoff_date = latest_date - pd.Timedelta(days=days)
        df_subset = df_all[df_all["伝票日付"] >= cutoff_date].copy()
        hol_max = df_subset["伝票日付"].max()
        hol_min = df_subset["伝票日付"].min()
        holidays = get_japanese_holidays(hol_min, hol_max)
        print("[DEBUG] df_subset_days=", df_subset["伝票日付"].nunique(), "range=", hol_min, "->", hol_max)
        try:
            actual, pred = full_walkforward(
                df_subset,
                holidays=holidays,
                df_reserve=df_reserve,
                min_stage1_days=MIN_STAGE1_DAYS,
                min_stage2_days=MIN_STAGE2_DAYS,
                top_n=2,
            )
            print(f"[DEBUG] returned_lengths actual={len(actual) if isinstance(actual, list) else 'NA'} pred={len(pred) if isinstance(pred, list) else 'NA'}")
            if isinstance(actual, list) and isinstance(pred, list) and len(actual) > 0 and len(pred) > 0:
                r2 = r2_score(actual, pred)
                mae = mean_absolute_error(actual, pred)
                results.append((days, r2, mae))
                print(f"✅ R² = {r2:.3f}, MAE = {mae:,.0f}kg")
            else:
                print("⚠ 評価に十分なデータがありません (予測件数0)")
                results.append((days, None, None))
        except Exception as e:
            print(f"❌ エラー: {e}")
            results.append((days, None, None))

    # 結果表示
    import pandas as pd

    df_result = pd.DataFrame(results, columns=["days", "R2", "MAE"])
    print("\n=== 評価結果 ===")
    print(df_result)
    if df_result["R2"].notna().any():
        plt.plot(df_result["days"], df_result["R2"], marker="o")
        plt.xlabel("Days")
        plt.ylabel("R² Score")
        plt.title("日数別 R² 評価 (new_model1)")
        plt.grid(True)
        plt.show()
else:
    print('Imports successful. To run pipeline, set RUN_PIPELINE = True in this cell.')


[DEBUG] ReserveFeatureBuilder.build: incoming columns=['予約日', '予約得意先名', '固定客', '予約台数']
[DEBUG] ReserveFeatureBuilder.build: sample rows=
{'予約日': [Timestamp('2023-01-04 00:00:00'), Timestamp('2023-01-04 00:00:00'), Timestamp('2023-01-04 00:00:00')], '予約得意先名': ['アンデス', 'リサイクルレスキュー', '山口興業'], '固定客': [False, False, False], '予約台数': [1.0, 1.0, 2.0]}
[DEBUG] reservation_filtered_days= 761
[DEBUG] reservation_span= 2023-01-04 00:00:00 -> 2025-05-26 00:00:00

=== 300日分のデータで評価中 ===
[DEBUG] df_subset_days= 280 range= 2024-07-30 00:00:00 -> 2025-05-26 00:00:00
▶️ full_walkforward 開始
[DEBUG] full_walkforward: df_raw.shape=(47504, 3), holidays_type=<class 'list'> df_reserve.shape=(45731, 4)
[DEBUG] WeightFeatureBuilder.build: past_raw.shape=(47504, 3), target_items=['混合廃棄物A', '混合廃棄物B']
[DEBUG] WeightFeatureBuilder.build: holidays type=<class 'list'>, len_or_none=20
[DEBUG] WeightFeatureBuilder.build: df_pivot.shape=(280, 226), df_feat.shape=(280, 17)
[DEBUG] ReserveFeatureBuilder.build: incoming col

## 天気追加モデル

In [ ]:
# new_model2: PRE_HANNNYU/scripts/new_model2 内のローカルモジュール利用 (再読込対応)
import sys, os, importlib
SCRIPT_ROOT = "/works/scripts"
if SCRIPT_ROOT not in sys.path:
    sys.path.append(SCRIPT_ROOT)
# 既存モジュールを再読込して最新パッチ反映
import new_model2.feature_builder as nm2_fb
import new_model2.predict_model_v4_2_4 as nm2_pred
importlib.reload(nm2_fb)
importlib.reload(nm2_pred)
from new_model2.predict_model_v4_2_4 import full_walkforward
from new_model2.feature_builder import WeatherFeatureBuilder, ReserveFeatureBuilder
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd
import matplotlib.pyplot as plt

print('[INFO] using WeatherFeatureBuilder from', WeatherFeatureBuilder.__module__)

# 予約 raw データは df_reserve として既に存在 (列: 予約日, 予約得意先名, 固定客, 予約台数)
# 集計済み特徴量はここでは不要なので生成しない（full_walkforward 内で再度 ReserveFeatureBuilder を用いるため）
if not pd.api.types.is_datetime64_any_dtype(df_reserve['予約日']):
    df_reserve['予約日'] = pd.to_datetime(df_reserve['予約日'])

# 日付型保証
if not pd.api.types.is_datetime64_any_dtype(df_all["伝票日付"]):
    df_all["伝票日付"] = pd.to_datetime(df_all["伝票日付"])

# 評価対象の日数リスト
days_list = [90,180,360,720]  # 検証を速くするため一旦 1 パターン
results = []

for days in days_list:
    print(f"\n=== {days}日分のデータで評価中 (weather) ===")
    latest_date = df_all["伝票日付"].max()
    cutoff_date = latest_date - pd.Timedelta(days=days)
    df_subset = df_all[df_all["伝票日付"] >= cutoff_date].copy()
    hol_min = df_subset["伝票日付"].min()
    hol_max = df_subset["伝票日付"].max()
    print('[DEBUG] hol_min, hol_max =', hol_min, hol_max)
    holidays = get_japanese_holidays(hol_min, hol_max)

    # 予約 raw サブセット（列 予約日 でフィルタ）
    mask = (df_reserve['予約日'] >= hol_min) & (df_reserve['予約日'] <= hol_max)
    df_reserve_raw_subset = df_reserve.loc[mask].copy()
    print(f"[DEBUG] reserve_raw_rows={len(df_reserve_raw_subset)} range={df_reserve_raw_subset['予約日'].min()}->{df_reserve_raw_subset['予約日'].max() if len(df_reserve_raw_subset) else 'NA'}")

    # 天気特徴量取得
    weather_builder = WeatherFeatureBuilder(start_date=hol_min, end_date=hol_max, enable_fallback=True)
    df_weather_full = weather_builder.build()
    df_weather = df_weather_full.loc[hol_min:hol_max].copy() if not df_weather_full.empty else df_weather_full
    if len(df_weather) > 0:
        print(f"[DEBUG] weather_rows={len(df_weather)} range={df_weather.index.min()}->{df_weather.index.max()}")
    else:
        print("[DEBUG] weather empty (fallback or no data)")

    try:
        actual, pred = full_walkforward(
            df_raw=df_subset,
            df_reserve=df_reserve_raw_subset,  # 修正: raw を渡す
            holidays=holidays,
            df_weather=df_weather,
            min_stage1_days=30,
            min_stage2_days=15,
            top_n=2,
        )
        print(f"[DEBUG] returned_lengths actual={len(actual) if isinstance(actual,list) else 'NA'} pred={len(pred) if isinstance(pred,list) else 'NA'}")
        if isinstance(actual, list) and isinstance(pred, list) and len(actual) > 0 and len(pred) > 0:
            r2 = r2_score(actual, pred)
            mae = mean_absolute_error(actual, pred)
            results.append((days, r2, mae))
            print(f"✅ R² = {r2:.3f}, MAE = {mae:,.0f}kg")
        else:
            print("⚠ 評価に十分なデータがありません (予測件数0)")
            results.append((days, None, None))
    except Exception as e:
        print(f"❌ エラー: {e}")
        results.append((days, None, None))

# 結果表示
import pandas as pd

df_result = pd.DataFrame(results, columns=["days", "R2", "MAE"])
print("\n=== 評価結果 (new_model2) ===")
print(df_result)
if df_result["R2"].notna().any():
    plt.plot(df_result["days"], df_result["R2"], marker="o")
    plt.xlabel("Days")
    plt.ylabel("R² Score")
    plt.title("日数別 R² 評価 (new_model2)")
    plt.grid(True)
    plt.show()
else:
    print("R² 有効値が無いためプロットをスキップ")

[INFO] using WeatherFeatureBuilder from new_model2.feature_builder

=== 90日分のデータで評価中 (weather) ===
[DEBUG] hol_min, hol_max = 2025-02-25 00:00:00 2025-05-26 00:00:00
[DEBUG] reserve_raw_rows=4773 range=2025-02-25 00:00:00->2025-05-26 00:00:00
[Weather] fetch 2025-02-25 -> 2025-05-26
[Weather] final params start_date=2025-02-25 end_date=2025-05-26
[Weather] status=200 url=https://archive-api.open-meteo.com/v1/archive?latitude=35.6895&longitude=139.6917&start_date=2025-02-25&end_date=2025-05-26&daily=temperature_2m_mean&daily=precipitation_sum&timezone=Asia%2FTokyo
[Weather] rows=91 cols=['平均気温', '降水量', '天気_大雨', '天気_晴れ', '天気_雨', '天気_台風']
[DEBUG] weather_rows=91 range=2025-02-25 00:00:00->2025-05-26 00:00:00
▶️ full_walkforward(new_model2) 開始
[DEBUG] target_items=['混合廃棄物A', '混合廃棄物B']
[DEBUG] feature_list_len=23 (orig=23) df_feat_rows=74
[DEBUG] dates_len=74 min_stage1_days=30 min_stage2_days=15
[SKIP] 2025-03-07 (i=0) < min_stage1_days=30
[SKIP] 2025-03-13 (i=5) < min_stage1_days=30
[SKIP

In [10]:
# デバッグ: hol_min / hol_max の型確認用一時セル
print('sample hol_min/hol_max (from df_all tail)')
print(df_all['伝票日付'].tail(1), type(df_all['伝票日付'].iloc[-1]))

sample hol_min/hol_max (from df_all tail)
184249   2025-05-26
Name: 伝票日付, dtype: datetime64[ns] <class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [11]:
# === 追加セル (このセルを特徴量削減セルより前に新規挿入) ===
# full_walkforward の現在シグネチャに合わせて余分な引数を自動的に除去し
# 戻り値を (actual, pred, model, dates) の4要素に正規化するラッパを直接適用します。
import sys
import inspect
import importlib

# パス設定を確実に行う
SCRIPT_ROOT = "/works/scripts"
if SCRIPT_ROOT not in sys.path:
    sys.path.insert(0, SCRIPT_ROOT)

# モジュールキャッシュをクリア
for name in list(sys.modules.keys()):
    if name.startswith('new_model2'):
        del sys.modules[name]

try:
    import new_model2.predict_model_v4_2_4 as _nm2p
    importlib.reload(_nm2p)
    print("[INFO] new_model2.predict_model_v4_2_4 successfully loaded and reloaded")
except Exception as e:
    print(f"[ERROR] Failed to load new_model2: {e}")
    # フォールバック: 前のセルで既に読み込まれたfull_walkforwardを使用
    from new_model2.predict_model_v4_2_4 import full_walkforward as _original_full_walkforward
    
    def _fw_wrapper(*args, **kwargs):
        # 引数フィルタリングなし（既存のfull_walkforwardをそのまま使用）
        res = _original_full_walkforward(*args, **kwargs)
        # 戻り値正規化
        if not isinstance(res, tuple):
            res = (res,)
        if len(res) == 2:
            actual, pred = res
            model = None
            dates = list(range(len(actual)))
        elif len(res) == 3:
            actual, pred, model = res
            dates = list(range(len(actual)))
        elif len(res) >= 4:
            actual, pred, model, dates = res[:4]
        else:
            actual, pred, model, dates = [], [], None, []
        return actual, pred, model, dates
    
    print("[INFO] Using fallback wrapper")
    # モジュール更新をスキップしてラッパーのみ適用
    import new_model2.predict_model_v4_2_4 as _nm2p
    _nm2p.full_walkforward = _fw_wrapper
    from new_model2.predict_model_v4_2_4 import full_walkforward
    print("[INFO] fallback wrapper installed (always returns 4要素)")
else:
    # 正常ロード時の処理
    _original_full_walkforward = _nm2p.full_walkforward
    _sig = inspect.signature(_original_full_walkforward)
    _supported = set(_sig.parameters.keys())
    print("[INFO] original full_walkforward signature:", _sig)

    def _fw_wrapper(*args, **kwargs):
        # 余分な引数を落とす
        filtered = {k:v for k,v in kwargs.items() if k in _supported}
        dropped = set(kwargs.keys()) - set(filtered.keys())
        if dropped:
            print(f"[WARN] 未サポート引数削除: {dropped}")
        res = _original_full_walkforward(*args, **filtered)
        # 戻り値正規化
        if not isinstance(res, tuple):
            res = (res,)
        if len(res) == 2:
            actual, pred = res
            model = None
            dates = list(range(len(actual)))
        elif len(res) == 3:
            actual, pred, model = res
            dates = list(range(len(actual)))
        elif len(res) >= 4:
            actual, pred, model, dates = res[:4]
        else:
            actual, pred, model, dates = [], [], None, []
        return actual, pred, model, dates

    # モンキーパッチ
    _nm2p.full_walkforward = _fw_wrapper
    from new_model2.predict_model_v4_2_4 import full_walkforward
    print("[INFO] wrapper installed (always returns 4要素)")

[INFO] new_model2.predict_model_v4_2_4 successfully loaded and reloaded
[INFO] original full_walkforward signature: (df_raw, holidays, df_reserve, df_weather, min_stage1_days, min_stage2_days, top_n=5, allowed_features=None)
[INFO] wrapper installed (always returns 4要素)


In [12]:
# === 365日実行結果の詳細分析 ===
print("=== 365日実行結果サマリー ===")
print(f"ベースラインMAE: {base_mae:.4f}")
print(f"予測回数: {len(base_err_df)}回")
print(f"データ期間: {hol_min} → {hol_max}")

if 'imp_all' in locals():
    print(f"\n=== 特徴量重要度トップ10 ===")
    print(imp_all.head(10))
    
    print(f"\n=== 削減候補特徴量（重要度下位5） ===")
    if len(reducible) >= 5:
        print(reducible.tail(5))
    else:
        print("削減候補が5個未満です")

if 'cand_mae' in locals():
    print(f"\n=== 削減テスト結果 ===")
    print(f"削除対象: {to_remove}")
    print(f"性能変化: {base_mae:.4f} → {cand_mae:.4f}")
    print(f"相対増加: {rel_inc*100:.2f}%")
    print(f"統計的有意性: p={p_value:.4f} ({method})")
    print(f"受容判定: {'ACCEPT' if accept else 'REJECT'}")
else:
    print("\n削減テストが未実行または失敗")

print(f"\n[INFO] 365日の長期データによる評価が完了しました")

=== 365日実行結果サマリー ===


NameError: name 'base_mae' is not defined

In [15]:
# --- 改良版 多段階削減ループ (true names) ---
from copy import deepcopy
import random
import math
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

DEF_REL_MAE_TOL = 0.005  # 0.5% 悪化まで許容
DEF_REMOVE_STEP = 1

PROTECT_EXACT = set()
PROTECT_PREFIXES = ("合計",)

random.seed(42)
np.random.seed(42)

# === baseline が未定義ならここで自動生成 ===
_need_baseline = False
for _v in ["base_actual", "base_pred", "base_model", "base_dates"]:
    if _v not in globals():
        _need_baseline = True
        break

if _need_baseline:
    print('[INFO] baseline 未定義 -> 自動計算開始')
    try:
        # 祝日リスト再生成（存在すれば再利用）
        if 'holidays' not in globals():
            hol_min = df_all['伝票日付'].min()
            hol_max = df_all['伝票日付'].max()
            holidays = get_japanese_holidays(hol_min, hol_max)
        # パラメータ既定
        TOP_N = globals().get('TOP_N', 2)
        MIN_STAGE1_DAYS = globals().get('MIN_STAGE1_DAYS', 30)
        MIN_STAGE2_DAYS = globals().get('MIN_STAGE2_DAYS', 15)
        # 天気特徴量存在確認
        if 'df_weather_full' not in globals():
            try:
                from new_model2.feature_builder import WeatherFeatureBuilder
                wb = WeatherFeatureBuilder(start_date=df_all['伝票日付'].min(), end_date=df_all['伝票日付'].max(), enable_fallback=True)
                df_weather_full = wb.build()
                print(f"[INFO] weather built rows={len(df_weather_full)}")
            except Exception as e:
                print('[WARN] 天気特徴量生成失敗 -> なしで継続', e)
                df_weather_full = None
        # full_walkforward import (既にラッパ適用セルがある前提)
        try:
            from scripts.new_model2.predict_model_v4_2_4 import full_walkforward, get_target_items, get_feature_list
        except Exception:
            from new_model2.predict_model_v4_2_4 import full_walkforward, get_target_items, get_feature_list
        base_actual, base_pred, base_model, base_dates = full_walkforward(
            df_all, holidays, df_reserve, df_weather_full, MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
            top_n=TOP_N, allowed_features=None
        )
        print(f"[INFO] baseline 再計算完了 len={len(base_actual)}")
    except Exception as e:
        print('[ERROR] baseline 自動計算失敗:', e)
        base_actual, base_pred, base_model, base_dates = [], [], None, []

if len(base_actual) == 0:
    print('[ABORT] baseline が空のため特徴量削減をスキップ')
else:
    base_err_df = pd.DataFrame({
        'date': base_dates,
        'actual': base_actual,
        'pred': base_pred
    })
    base_err_df['abs_err'] = (base_err_df['actual'] - base_err_df['pred']).abs()
    base_mae = base_err_df['abs_err'].mean()
    base_r2 = r2_score(base_actual, base_pred) if len(base_actual)>1 else float('nan')
    print(f"[BASE] MAE={base_mae:,.2f} R2={base_r2:.3f} n={len(base_actual)}")

    # 重要度抽出
    imp_true = extract_true_feature_importances(base_model)
    all_features_ordered = imp_true['feature'].tolist()
    print(f"[INFO] feature count={len(all_features_ordered)}")

    # 保護対象除外
    reducible_df = imp_true[~imp_true['feature'].isin(PROTECT_EXACT)]
    for p in PROTECT_PREFIXES:
        reducible_df = reducible_df[~reducible_df['feature'].str.startswith(p)]

    # 重要度小さい順に並べ替え
    reducible_df = reducible_df.sort_values('abs_coef', ascending=True).reset_index(drop=True)

    current_keep = all_features_ordered.copy()
    history = []
    max_steps = 30

    # 安全のため TOP_N など再取得
    TOP_N = globals().get('TOP_N', 2)
    MIN_STAGE1_DAYS = globals().get('MIN_STAGE1_DAYS', 30)
    MIN_STAGE2_DAYS = globals().get('MIN_STAGE2_DAYS', 15)

    for step in range(max_steps):
        cand_remove = []
        for f in reducible_df['feature']:
            if f in current_keep and f not in PROTECT_EXACT and not any(f.startswith(p) for p in PROTECT_PREFIXES):
                cand_remove.append(f)
            if len(cand_remove) >= DEF_REMOVE_STEP:
                break
        if not cand_remove:
            print('[END] 除去候補なし')
            break

        tentative = [f for f in current_keep if f not in cand_remove]

        try:
            original_list = get_feature_list(get_target_items(df_all, TOP_N), extra_features=["天気_晴れ","天気_雨","天気_大雨","天気_台風"])
            tentative = [f for f in tentative if f in original_list]
        except Exception as e:
            print('[WARN] original_list取得失敗 -> スキップ', e)
        if len(tentative) == 0:
            print(f"[SKIP] step={step} 交差後特徴量ゼロ -> 中断")
            break

        print(f"\n[TRY] step={step} remove={cand_remove} -> tentative_len={len(tentative)}")

        cand_actual, cand_pred, cand_model, cand_dates = full_walkforward(
            df_all, holidays, df_reserve, df_weather_full, MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
            top_n=TOP_N, allowed_features=tentative
        )
        if len(cand_actual) < 3:
            print('[REJECT] データ不足')
            history.append({'step': step, 'removed': cand_remove, 'result': 'REJECT_DATA'})
            break

        cand_err = pd.DataFrame({'actual': cand_actual, 'pred': cand_pred})
        cand_err['abs_err'] = (cand_err['actual'] - cand_err['pred']).abs()
        cand_mae = cand_err['abs_err'].mean()
        cand_r2 = r2_score(cand_actual, cand_pred) if len(cand_actual)>1 else float('nan')

        rel_diff = (cand_mae - base_mae) / base_mae

        base_df_j = base_err_df.copy()
        base_df_j['date'] = base_dates
        cand_df_j = cand_err.copy()
        cand_df_j['date'] = cand_dates
        merged = base_df_j.merge(cand_df_j[['date','abs_err']], on='date', suffixes=('_base','_cand'))
        p_value = np.nan
        if len(merged) >= 5:
            try:
                from scipy.stats import wilcoxon
                stat, p_value = wilcoxon(merged['abs_err_base'], merged['abs_err_cand'])
            except Exception as e:
                print('[WARN] wilcoxon失敗', e)

        accept = (rel_diff <= DEF_REL_MAE_TOL) and (math.isnan(p_value) or p_value > 0.05)

        print(f"[EVAL] cand_mae={cand_mae:,.2f} (diff={rel_diff*100:.2f}%) R2={cand_r2:.3f} p={p_value if not math.isnan(p_value) else 'NA'} -> {'ACCEPT' if accept else 'REJECT'}")

        history.append({
            'step': step,
            'removed': cand_remove,
            'cand_mae': cand_mae,
            'cand_r2': cand_r2,
            'base_mae': base_mae,
            'base_r2': base_r2,
            'rel_diff': rel_diff,
            'p_value': p_value,
            'accept': accept
        })

        if accept:
            current_keep = tentative
            base_actual, base_pred, base_model, base_dates = cand_actual, cand_pred, cand_model, cand_dates
            base_err_df = cand_err.copy()
            base_err_df['date'] = base_dates
            base_mae = cand_mae
            base_r2 = cand_r2
            reducible_df = reducible_df[~reducible_df['feature'].isin(cand_remove)].reset_index(drop=True)
        else:
            PROTECT_EXACT.update(cand_remove)
            reducible_df = reducible_df[~reducible_df['feature'].isin(PROTECT_EXACT)].reset_index(drop=True)

        if len(reducible_df) == 0:
            print('[END] これ以上削減不可')
            break

    print('\n--- 削減履歴 ---')
    try:
        display(pd.DataFrame(history))
    except Exception:
        print(pd.DataFrame(history))

[INFO] baseline 未定義 -> 自動計算開始
▶️ full_walkforward(new_model2) 開始
[DEBUG] target_items=['混合廃棄物A', '混合廃棄物B']
[DEBUG] feature_list_len=23 (orig=23) df_feat_rows=751
[DEBUG] dates_len=751 min_stage1_days=20 min_stage2_days=10
[SKIP] 2023-01-14 (i=0) < min_stage1_days=20
[SKIP] 2023-01-19 (i=5) < min_stage1_days=20
[SKIP] 2023-01-24 (i=10) < min_stage1_days=20
[SKIP] 2023-01-29 (i=15) < min_stage1_days=20

=== 2023-02-03 を予測中 (i=20) ===
[DEBUG] feature_list_len=23 (orig=23) df_feat_rows=751
[DEBUG] dates_len=751 min_stage1_days=20 min_stage2_days=10
[SKIP] 2023-01-14 (i=0) < min_stage1_days=20
[SKIP] 2023-01-19 (i=5) < min_stage1_days=20
[SKIP] 2023-01-24 (i=10) < min_stage1_days=20
[SKIP] 2023-01-29 (i=15) < min_stage1_days=20

=== 2023-02-03 を予測中 (i=20) ===
[DEBUG] ステージ2未実行 rows=1/11

=== 2023-02-04 を予測中 (i=21) ===
[DEBUG] ステージ2未実行 rows=1/11

=== 2023-02-04 を予測中 (i=21) ===
[DEBUG] ステージ2未実行 rows=2/11

=== 2023-02-05 を予測中 (i=22) ===
[DEBUG] ステージ2未実行 rows=2/11

=== 2023-02-05 を予測中 (i=22) ===

NameError: name 'extract_true_feature_importances' is not defined

In [3]:
# === 最終特徴量サマリー & 再学習・保存ユーティリティ ===
import os, json, pickle, time
from datetime import datetime

# current_keep が存在し、最低限の特徴があるか確認
if 'current_keep' not in globals() or not current_keep:
    print('[FINAL] current_keep が未定義または空のためスキップ')
else:
    print(f'[FINAL] 最終特徴量数: {len(current_keep)}')
    print(current_keep[:30] + (['...'] if len(current_keep) > 30 else []))

    # 祝日と天気を再利用 / 無ければ再生成
    try:
        holidays
    except NameError:
        hol_min = df_all['伝票日付'].min(); hol_max = df_all['伝票日付'].max()
        holidays = get_japanese_holidays(hol_min, hol_max)
    try:
        df_weather_full
    except NameError:
        df_weather_full = None

    # パラメータ取得（存在しなければデフォルト）
    TOP_N = globals().get('TOP_N', 2)
    MIN_STAGE1_DAYS = globals().get('MIN_STAGE1_DAYS', 30)
    MIN_STAGE2_DAYS = globals().get('MIN_STAGE2_DAYS', 15)

    # full_walkforward import（ラッパ適用済み想定）
    try:
        from scripts.new_model2.predict_model_v4_2_4 import full_walkforward
    except Exception:
        from new_model2.predict_model_v4_2_4 import full_walkforward

    print('[FINAL] 最終特徴量で再学習開始')
    t0 = time.time()
    f_actual, f_pred, f_model, f_dates = full_walkforward(
        df_all, holidays, df_reserve, df_weather_full,
        MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
        top_n=TOP_N, allowed_features=current_keep
    )
    if len(f_actual) == 0:
        print('[FINAL][ERROR] 再学習結果が空')
    else:
        from sklearn.metrics import r2_score, mean_absolute_error
        f_mae = mean_absolute_error(f_actual, f_pred)
        f_r2 = r2_score(f_actual, f_pred) if len(f_actual) > 1 else float('nan')
        print(f'[FINAL] 再学習完了 MAE={f_mae:,.2f} R2={f_r2:.3f} n={len(f_actual)} elapsed={(time.time()-t0):.1f}s')

        # 出力ディレクトリ
        out_dir = '/home/ken/ai_work/works/data'
        os.makedirs(out_dir, exist_ok=True)

        # 特徴量リスト保存
        feat_path = os.path.join(out_dir, 'selected_features_final.txt')
        with open(feat_path, 'w', encoding='utf-8') as f:
            for feat in current_keep:
                f.write(feat + '\n')
        print('[FINAL] 特徴量リスト保存:', feat_path)

        # メタ情報 + 成果指標
        meta = {
            'generated_at': datetime.utcnow().isoformat() + 'Z',
            'feature_count': len(current_keep),
            'mae': f_mae,
            'r2': f_r2,
            'n_predictions': len(f_actual),
            'params': {
                'TOP_N': TOP_N,
                'MIN_STAGE1_DAYS': MIN_STAGE1_DAYS,
                'MIN_STAGE2_DAYS': MIN_STAGE2_DAYS,
                'REL_MAE_TOL': DEF_REL_MAE_TOL,
                'REMOVE_STEP': DEF_REMOVE_STEP
            }
        }
        meta_path = os.path.join(out_dir, 'final_model_meta.json')
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        print('[FINAL] メタ情報保存:', meta_path)

        # モデル保存（API推論対応: scikit-learn Estimatorのみ保存）
        try:
            # f_modelがdict型の場合は、推論用モデル本体（例: f_model['model']）を保存
            model_to_save = f_model['model'] if isinstance(f_model, dict) and 'model' in f_model else f_model
            # APIでpredictできるか確認
            if hasattr(model_to_save, 'predict'):
                model_path = os.path.join(out_dir, 'final_stage1_model.pkl')
                with open(model_path, 'wb') as f:
                    pickle.dump(model_to_save, f)
                print('[FINAL] 推論対応モデル保存:', model_path)
            else:
                print('[FINAL][ERROR] 保存対象が推論対応モデルではありません:', type(model_to_save))
        except Exception as e:
            print('[FINAL][WARN] モデル保存失敗:', e)

        # 簡易差分 (baseline との差) があれば表示
        try:
            diff_mae = (f_mae - base_mae) / base_mae * 100
            print(f'[FINAL] baselineとの差: MAE差分={diff_mae:.2f}%')
        except Exception:
            pass


[FINAL] current_keep が未定義または空のためスキップ


In [1]:
# モデル互換性テスト & ローカルマウント確認
import os
import pickle

# モデルファイルのパス
model_path = '/works/data/final_stage1_model.pkl'

# ローカルマウント確認（ファイル存在チェック）
if os.path.exists(model_path):
    print(f'[OK] モデルファイルが見つかりました: {model_path}')
else:
    print(f'[NG] モデルファイルが見つかりません: {model_path}')

try:
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    print('[OK] モデルのロードに成功しました。')
    # ダミーデータで推論（shapeはモデルに合わせて調整してください）
    import numpy as np
    dummy = np.zeros((1, len(getattr(model, "feature_names_in_", [0]*10))))
    try:
        pred = model.predict(dummy)
        print('[OK] 推論も成功しました。予測値:', pred)
    except Exception as e:
        print('[NG] 推論時エラー:', e)
except Exception as e:
    print('[NG] モデルロード時エラー:', e)

# /works/data/input ディレクトリのファイル一覧表示でマウント状況確認
input_dir = '/works/data/input'
if os.path.exists(input_dir):
    files = os.listdir(input_dir)
    print(f'[OK] 入力ディレクトリが見つかりました: {input_dir}')
    print('ファイル一覧:', files)
else:
    print(f'[NG] 入力ディレクトリが見つかりません: {input_dir}')

[NG] モデルファイルが見つかりません: /works/data/final_stage1_model.pkl
[NG] モデルロード時エラー: [Errno 2] No such file or directory: '/works/data/final_stage1_model.pkl'
[NG] 入力ディレクトリが見つかりません: /works/data/input


In [2]:
# WSL環境用: 正しいパスでモデル互換性テスト & マウント確認
import os
import pickle

# モデルファイルのパス（WSLの実際のパスに修正）
model_path = '/home/ken/ai_work/works/data/final_stage1_model.pkl'

# ローカルマウント確認（ファイル存在チェック）
if os.path.exists(model_path):
    print(f'[OK] モデルファイルが見つかりました: {model_path}')
else:
    print(f'[NG] モデルファイルが見つかりません: {model_path}')

try:
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    print('[OK] モデルのロードに成功しました。')
    # ダミーデータで推論（shapeはモデルに合わせて調整してください）
    import numpy as np
    dummy = np.zeros((1, len(getattr(model, "feature_names_in_", [0]*10))))
    try:
        pred = model.predict(dummy)
        print('[OK] 推論も成功しました。予測値:', pred)
    except Exception as e:
        print('[NG] 推論時エラー:', e)
except Exception as e:
    print('[NG] モデルロード時エラー:', e)

# 入力ディレクトリのファイル一覧表示でマウント状況確認
input_dir = '/home/ken/ai_work/works/data/input'
if os.path.exists(input_dir):
    files = os.listdir(input_dir)
    print(f'[OK] 入力ディレクトリが見つかりました: {input_dir}')
    print('ファイル一覧:', files)
else:
    print(f'[NG] 入力ディレクトリが見つかりません: {input_dir}')

[OK] モデルファイルが見つかりました: /home/ken/ai_work/works/data/final_stage1_model.pkl
[OK] モデルのロードに成功しました。
[NG] 推論時エラー: 'dict' object has no attribute 'predict'
[OK] 入力ディレクトリが見つかりました: /home/ken/ai_work/works/data/input
ファイル一覧: ['2020顧客.csv', '受入_時刻', 'yoyaku_data.csv', '2023_all.csv\uf03aZone.Identifier', '2023_all.csv', '2020顧客.csv\uf03aZone.Identifier', 'yoyaku_data.csv\uf03aZone.Identifier', '2021顧客.csv\uf03aZone.Identifier', '20240501-20250422.csv\uf03aZone.Identifier', '2021顧客.csv', '2022顧客.csv', '20240501-20250422.csv', '2022顧客.csv\uf03aZone.Identifier']


In [13]:
# --- モデルをpredict機能付きで再保存 ---
import os
import pickle

# 元モデルファイルのパス
old_model_path = '/work/data/final_stage1_model.pkl'
new_model_path = '/work/data/final_stage1_model_predictable.pkl'

# モデルをロード
with open(old_model_path, 'rb') as f:
    model = pickle.load(f)

# predictメソッドがなければラップ
if not hasattr(model, 'predict'):
    from sklearn.base import BaseEstimator
    class PredictableModel(BaseEstimator):
        def __init__(self, base_model):
            self.base_model = base_model
        def predict(self, X):
            # base_modelがpredict_probaやtransformしか持たない場合は適宜修正
            if hasattr(self.base_model, 'predict'):
                return self.base_model.predict(X)
            elif hasattr(self.base_model, 'transform'):
                return self.base_model.transform(X)
            else:
                raise AttributeError('base_modelにpredict/transformがありません')
        def __getattr__(self, name):
            return getattr(self.base_model, name)
    model = PredictableModel(model)

# 新しいファイル名で再保存
with open(new_model_path, 'wb') as f:
    pickle.dump(model, f)
print(f'[OK] predict機能付きで再保存しました: {new_model_path}')

[OK] predict機能付きで再保存しました: /work/data/final_stage1_model_predictable.pkl


/usr/local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator ElasticNet from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator VarianceThreshold from version 1.6.1 when using version 1.7.2. This might lead to breaking code or i

In [14]:
# --- 再学習とpredict機能付きで再保存 ---
import os, pickle, time
from datetime import datetime
from sklearn.metrics import r2_score, mean_absolute_error

# 必要なデータ・パラメータの準備
# df_all, df_reserve, holidays, df_weather_full, current_keep などは既存セルで定義済み前提
TOP_N = globals().get('TOP_N', 2)
MIN_STAGE1_DAYS = globals().get('MIN_STAGE1_DAYS', 30)
MIN_STAGE2_DAYS = globals().get('MIN_STAGE2_DAYS', 15)

# full_walkforwardのimport（ラッパー適用済み想定）
try:
    from scripts.new_model2.predict_model_v4_2_4 import full_walkforward
except Exception:
    from new_model2.predict_model_v4_2_4 import full_walkforward

print('[INFO] 最終特徴量で再学習開始')
t0 = time.time()
f_actual, f_pred, f_model, f_dates = full_walkforward(
    df_all, holidays, df_reserve, df_weather_full,
    MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
    top_n=TOP_N, allowed_features=current_keep
)
if len(f_actual) == 0:
    print('[ERROR] 再学習結果が空です')
else:
    f_mae = mean_absolute_error(f_actual, f_pred)
    f_r2 = r2_score(f_actual, f_pred) if len(f_actual) > 1 else float('nan')
    print(f'[OK] 再学習完了 MAE={f_mae:,.2f} R2={f_r2:.3f} n={len(f_actual)} elapsed={(time.time()-t0):.1f}s')

    # モデル保存（predict機能付きでラップ）
    out_dir = '/work/data'
    os.makedirs(out_dir, exist_ok=True)
    model_path = os.path.join(out_dir, 'final_stage1_model_predictable.pkl')
    model_to_save = f_model['model'] if isinstance(f_model, dict) and 'model' in f_model else f_model
    if not hasattr(model_to_save, 'predict'):
        from sklearn.base import BaseEstimator
        class PredictableModel(BaseEstimator):
            def __init__(self, base_model):
                self.base_model = base_model
            def predict(self, X):
                if hasattr(self.base_model, 'predict'):
                    return self.base_model.predict(X)
                elif hasattr(self.base_model, 'transform'):
                    return self.base_model.transform(X)
                else:
                    raise AttributeError('base_modelにpredict/transformがありません')
            def __getattr__(self, name):
                return getattr(self.base_model, name)
        model_to_save = PredictableModel(model_to_save)
    with open(model_path, 'wb') as f:
        pickle.dump(model_to_save, f)
    print(f'[OK] predict機能付きで再保存しました: {model_path}')

[INFO] 最終特徴量で再学習開始


NameError: name 'holidays' is not defined

In [15]:
# --- holidays未定義エラー対策: 祝日リストを再生成して再学習・再保存 ---
from datetime import datetime

if 'holidays' not in globals():
    hol_min = df_all['伝票日付'].min()
    hol_max = df_all['伝票日付'].max()
    holidays = get_japanese_holidays(hol_min, hol_max)
    print(f'[INFO] holidays再生成: {hol_min} → {hol_max} ({len(holidays)}日)')

# 再学習とpredict機能付きで再保存（前セルと同じ処理）
TOP_N = globals().get('TOP_N', 2)
MIN_STAGE1_DAYS = globals().get('MIN_STAGE1_DAYS', 30)
MIN_STAGE2_DAYS = globals().get('MIN_STAGE2_DAYS', 15)
try:
    from scripts.new_model2.predict_model_v4_2_4 import full_walkforward
except Exception:
    from new_model2.predict_model_v4_2_4 import full_walkforward

print('[INFO] 最終特徴量で再学習開始')
t0 = time.time()
f_actual, f_pred, f_model, f_dates = full_walkforward(
    df_all, holidays, df_reserve, df_weather_full,
    MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
    top_n=TOP_N, allowed_features=current_keep
)
if len(f_actual) == 0:
    print('[ERROR] 再学習結果が空です')
else:
    from sklearn.metrics import r2_score, mean_absolute_error
    f_mae = mean_absolute_error(f_actual, f_pred)
    f_r2 = r2_score(f_actual, f_pred) if len(f_actual) > 1 else float('nan')
    print(f'[OK] 再学習完了 MAE={f_mae:,.2f} R2={f_r2:.3f} n={len(f_actual)} elapsed={(time.time()-t0):.1f}s')

    # モデル保存（predict機能付きでラップ）
    out_dir = '/work/data'
    os.makedirs(out_dir, exist_ok=True)
    model_path = os.path.join(out_dir, 'final_stage1_model_predictable.pkl')
    model_to_save = f_model['model'] if isinstance(f_model, dict) and 'model' in f_model else f_model
    if not hasattr(model_to_save, 'predict'):
        from sklearn.base import BaseEstimator
        class PredictableModel(BaseEstimator):
            def __init__(self, base_model):
                self.base_model = base_model
            def predict(self, X):
                if hasattr(self.base_model, 'predict'):
                    return self.base_model.predict(X)
                elif hasattr(self.base_model, 'transform'):
                    return self.base_model.transform(X)
                else:
                    raise AttributeError('base_modelにpredict/transformがありません')
            def __getattr__(self, name):
                return getattr(self.base_model, name)
        model_to_save = PredictableModel(model_to_save)
    with open(model_path, 'wb') as f:
        pickle.dump(model_to_save, f)
    print(f'[OK] predict機能付きで再保存しました: {model_path}')

[INFO] holidays再生成: 2020-01-04 00:00:00 → 2025-05-26 00:00:00 (99日)
[INFO] 最終特徴量で再学習開始


NameError: name 'df_weather_full' is not defined

In [20]:
# --- extract_true_feature_importances未定義エラー対策: 代替で特徴量リストを取得して再学習・再保存 ---
# f_modelから直接特徴量リストを取得する方法（scikit-learn系モデルの場合）
if 'f_model' in globals():
    try:
        # scikit-learnのモデルならfeature_names_in_属性を利用
        if hasattr(f_model, 'feature_names_in_'):
            current_keep = list(f_model.feature_names_in_)
            print(f'[INFO] current_keepをfeature_names_in_から取得: {len(current_keep)} features')
        # dict型でmodel本体がある場合
        elif isinstance(f_model, dict) and 'model' in f_model and hasattr(f_model['model'], 'feature_names_in_'):
            current_keep = list(f_model['model'].feature_names_in_)
            print(f'[INFO] current_keepをdict型model.feature_names_in_から取得: {len(current_keep)} features')
        else:
            print('[ERROR] f_modelから特徴量リスト取得不可')
            current_keep = None
    except Exception as e:
        print('[ERROR] current_keep取得失敗:', e)
        current_keep = None
else:
    print('[ERROR] f_modelが未定義です')
    current_keep = None

# 再学習とpredict機能付きで再保存（current_keepが定義された場合のみ）
if current_keep:
    print('[INFO] 直近365日分で再学習開始（current_keep: feature_names_in_ベース）')
    t0 = time.time()
    f_actual, f_pred, f_model, f_dates = full_walkforward(
        df_recent, holidays, df_reserve_recent, df_weather_full,
        MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
        top_n=TOP_N, allowed_features=current_keep
    )
    if len(f_actual) == 0:
        print('[ERROR] 再学習結果が空です')
    else:
        from sklearn.metrics import r2_score, mean_absolute_error
        f_mae = mean_absolute_error(f_actual, f_pred)
        f_r2 = r2_score(f_actual, f_pred) if len(f_actual) > 1 else float('nan')
        print(f'[OK] 再学習完了 MAE={f_mae:,.2f} R2={f_r2:.3f} n={len(f_actual)} elapsed={(time.time()-t0):.1f}s')

        # モデル保存（predict機能付きでラップ）
        out_dir = '/work/data'
        os.makedirs(out_dir, exist_ok=True)
        model_path = os.path.join(out_dir, 'final_stage1_model_predictable_365days.pkl')
        model_to_save = f_model['model'] if isinstance(f_model, dict) and 'model' in f_model else f_model
        if not hasattr(model_to_save, 'predict'):
            from sklearn.base import BaseEstimator
            class PredictableModel(BaseEstimator):
                def __init__(self, base_model):
                    self.base_model = base_model
                def predict(self, X):
                    if hasattr(self.base_model, 'predict'):
                        return self.base_model.predict(X)
                    elif hasattr(self.base_model, 'transform'):
                        return self.base_model.transform(X)
                    else:
                        raise AttributeError('base_modelにpredict/transformがありません')
                def __getattr__(self, name):
                    return getattr(self.base_model, name)
            model_to_save = PredictableModel(model_to_save)
        with open(model_path, 'wb') as f:
            pickle.dump(model_to_save, f)
        print(f'[OK] 直近365日分でpredict機能付き再保存: {model_path}')
else:
    print('[ERROR] current_keepが定義できませんでした。f_modelの型や属性を確認してください。')

[ERROR] f_modelが未定義です
[ERROR] current_keepが定義できませんでした。f_modelの型や属性を確認してください。


In [ ]:
# --- f_model未定義エラー対策: allowed_features無しで一度学習しf_modelを取得、その後再学習・再保存 ---
print('[INFO] f_model未定義のため、まずallowed_features無しで学習してf_modelを取得します')
t0 = time.time()
f_actual_tmp, f_pred_tmp, f_model_tmp, f_dates_tmp = full_walkforward(
    df_recent, holidays, df_reserve_recent, df_weather_full,
    MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
    top_n=TOP_N, allowed_features=None
)
if f_model_tmp is None:
    print('[ERROR] f_model取得失敗。full_walkforwardの戻り値を確認してください。')
else:
    # f_modelを取得し、特徴量リストを抽出
    if hasattr(f_model_tmp, 'feature_names_in_'):
        current_keep = list(f_model_tmp.feature_names_in_)
        print(f'[INFO] current_keepをfeature_names_in_から取得: {len(current_keep)} features')
    elif isinstance(f_model_tmp, dict) and 'model' in f_model_tmp and hasattr(f_model_tmp['model'], 'feature_names_in_'):
        current_keep = list(f_model_tmp['model'].feature_names_in_)
        print(f'[INFO] current_keepをdict型model.feature_names_in_から取得: {len(current_keep)} features')
    else:
        print('[ERROR] f_model_tmpから特徴量リスト取得不可')
        current_keep = None
    # 再学習とpredict機能付きで再保存
    if current_keep:
        print('[INFO] 直近365日分で再学習開始（current_keep: feature_names_in_ベース）')
        t0 = time.time()
        f_actual, f_pred, f_model, f_dates = full_walkforward(
            df_recent, holidays, df_reserve_recent, df_weather_full,
            MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
            top_n=TOP_N, allowed_features=current_keep
        )
        if len(f_actual) == 0:
            print('[ERROR] 再学習結果が空です')
        else:
            from sklearn.metrics import r2_score, mean_absolute_error
            f_mae = mean_absolute_error(f_actual, f_pred)
            f_r2 = r2_score(f_actual, f_pred) if len(f_actual) > 1 else float('nan')
            print(f'[OK] 再学習完了 MAE={f_mae:,.2f} R2={f_r2:.3f} n={len(f_actual)} elapsed={(time.time()-t0):.1f}s')

            # モデル保存（predict機能付きでラップ）
            out_dir = '/work/data'
            os.makedirs(out_dir, exist_ok=True)
            model_path = os.path.join(out_dir, 'final_stage1_model_predictable_365days.pkl')
            model_to_save = f_model['model'] if isinstance(f_model, dict) and 'model' in f_model else f_model
            if not hasattr(model_to_save, 'predict'):
                from sklearn.base import BaseEstimator
                class PredictableModel(BaseEstimator):
                    def __init__(self, base_model):
                        self.base_model = base_model
                    def predict(self, X):
                        if hasattr(self.base_model, 'predict'):
                            return self.base_model.predict(X)
                        elif hasattr(self.base_model, 'transform'):
                            return self.base_model.transform(X)
                        else:
                            raise AttributeError('base_modelにpredict/transformがありません')
                    def __getattr__(self, name):
                        return getattr(self.base_model, name)
                model_to_save = PredictableModel(model_to_save)
            with open(model_path, 'wb') as f:
                pickle.dump(model_to_save, f)
            print(f'[OK] 直近365日分でpredict機能付き再保存: {model_path}')
    else:
        print('[ERROR] current_keepが定義できませんでした。f_model_tmpの型や属性を確認してください。')

In [22]:
# --- selected_features_final.txtを使用して再学習・評価・保存（高速化）---
import os, pickle, time
import pandas as pd
from datetime import datetime
from sklearn.metrics import r2_score, mean_absolute_error

# 特徴量リストをファイルから読み込み
feat_path = '/work/data/selected_features_final.txt'
with open(feat_path, 'r', encoding='utf-8') as f:
    current_keep = [line.strip() for line in f if line.strip()]
print(f'[INFO] 特徴量リスト読込: {len(current_keep)} features')

# 直近365日分のデータに絞り込み
latest_date = df_all['伝票日付'].max()
cutoff_date = latest_date - pd.Timedelta(days=365)
df_recent = df_all[df_all['伝票日付'] >= cutoff_date].copy()
print(f'[INFO] 直近365日分データ: {cutoff_date} → {latest_date} 件数={len(df_recent)}')

# holidays, df_weather_fullも直近365日に合わせて再生成
hol_min = df_recent['伝票日付'].min()
hol_max = df_recent['伝票日付'].max()
holidays = get_japanese_holidays(hol_min, hol_max)
try:
    from new_model2.feature_builder import WeatherFeatureBuilder
    wb = WeatherFeatureBuilder(start_date=hol_min, end_date=hol_max, enable_fallback=True)
    df_weather_full = wb.build()
    print(f'[INFO] df_weather_full再生成: {hol_min} → {hol_max} ({len(df_weather_full)}日)')
except Exception as e:
    print('[WARN] 天気特徴量生成失敗 -> なしで継続', e)
    df_weather_full = None

# 予約データも期間でフィルタ
if 'df_reserve' in globals():
    mask = (df_reserve['予約日'] >= hol_min) & (df_reserve['予約日'] <= hol_max)
    df_reserve_recent = df_reserve.loc[mask].copy()
else:
    df_reserve_recent = None

# 再学習・評価・保存
TOP_N = 2
MIN_STAGE1_DAYS = 30
MIN_STAGE2_DAYS = 15
try:
    from scripts.new_model2.predict_model_v4_2_4 import full_walkforward
except Exception:
    from new_model2.predict_model_v4_2_4 import full_walkforward

print('[INFO] selected_features_final.txtで再学習開始（高速化）')
t0 = time.time()
f_actual, f_pred, f_model, f_dates = full_walkforward(
    df_recent, holidays, df_reserve_recent, df_weather_full,
    MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
    top_n=TOP_N, allowed_features=current_keep
)
if len(f_actual) == 0:
    print('[ERROR] 再学習結果が空です')
else:
    f_mae = mean_absolute_error(f_actual, f_pred)
    f_r2 = r2_score(f_actual, f_pred) if len(f_actual) > 1 else float('nan')
    print(f'[OK] 再学習完了 MAE={f_mae:,.2f} R2={f_r2:.3f} n={len(f_actual)} elapsed={(time.time()-t0):.1f}s')

    # モデル保存（predict機能付きでラップ）
    out_dir = '/work/data'
    os.makedirs(out_dir, exist_ok=True)
    model_path = os.path.join(out_dir, 'final_stage1_model_predictable_365days_selected.pkl')
    model_to_save = f_model['model'] if isinstance(f_model, dict) and 'model' in f_model else f_model
    if not hasattr(model_to_save, 'predict'):
        from sklearn.base import BaseEstimator
        class PredictableModel(BaseEstimator):
            def __init__(self, base_model):
                self.base_model = base_model
            def predict(self, X):
                if hasattr(self.base_model, 'predict'):
                    return self.base_model.predict(X)
                elif hasattr(self.base_model, 'transform'):
                    return self.base_model.transform(X)
                else:
                    raise AttributeError('base_modelにpredict/transformがありません')
            def __getattr__(self, name):
                return getattr(self.base_model, name)
        model_to_save = PredictableModel(model_to_save)
    with open(model_path, 'wb') as f:
        pickle.dump(model_to_save, f)
    print(f'[OK] selected_features_final.txtでpredict機能付き再保存: {model_path}')

[INFO] 2025-09-12 10:30:00 new_model2.walkforward: ▶ full_walkforward(new_model2) start top_n=2 allowed_mode=whitelist


[INFO] 2025-09-12 10:30:00 new_model2.walkforward: [INIT] target_items=['混合廃棄物A', '混合廃棄物B']


[INFO] 特徴量リスト読込: 15 features
[INFO] 直近365日分データ: 2024-05-26 00:00:00 → 2025-05-26 00:00:00 件数=57297
[WARN] 天気特徴量生成失敗 -> なしで継続 No module named 'new_model2'
[INFO] selected_features_final.txtで再学習開始（高速化）


[INFO] 2025-09-12 10:30:00 new_model2.walkforward: [FEATURES] use=15 (orig=23) mode=whitelist
[INFO] 2025-09-12 10:30:00 new_model2.walkforward: [CONFIG] dates=342 min_stage1_days=30 min_stage2_days=15 eff_stage2_rows=15
[INFO] 2025-09-12 10:30:00 new_model2.walkforward: [CONFIG] dates=342 min_stage1_days=30 min_stage2_days=15 eff_stage2_rows=15


TypeError: '<=' not supported between instances of 'numpy.ndarray' and 'Timestamp'

In [24]:
# --- df_weather_fullのindex型修正（DatetimeIndexへ変換）---
if 'df_weather_full' in globals() and df_weather_full is not None:
    try:
        df_weather_full.index = pd.to_datetime(df_weather_full.index)
        print('[INFO] df_weather_full.indexをDatetimeIndexに変換しました')
    except Exception as e:
        print('[WARN] df_weather_full.index型変換失敗:', e)
else:
    print('[WARN] df_weather_fullが未定義またはNoneです')

[WARN] df_weather_fullが未定義またはNoneです


In [25]:
# --- df_weather_fullがNoneの場合の再生成とindex型修正（完全版）---
if 'df_weather_full' not in globals() or df_weather_full is None:
    hol_min = df_recent['伝票日付'].min()
    hol_max = df_recent['伝票日付'].max()
    try:
        from new_model2.feature_builder import WeatherFeatureBuilder
        wb = WeatherFeatureBuilder(start_date=hol_min, end_date=hol_max, enable_fallback=True)
        df_weather_full = wb.build()
        print(f'[INFO] df_weather_full再生成: {hol_min} → {hol_max} ({len(df_weather_full)}日)')
    except Exception as e:
        print('[WARN] 天気特徴量生成失敗 -> なしで継続', e)
        df_weather_full = None
if df_weather_full is not None and not isinstance(df_weather_full.index, pd.DatetimeIndex):
    df_weather_full = df_weather_full.copy()
    df_weather_full.index = pd.to_datetime(df_weather_full.index)
    print('[INFO] df_weather_full.indexを再度DatetimeIndexに変換しました')

[WARN] 天気特徴量生成失敗 -> なしで継続 No module named 'new_model2'


In [26]:
# 修復セル: import パス調整 ＋ new_model2 の import 検証 ＋ 天気特徴量の再生成と学習実行（高速）
import sys, os, pandas as pd, numpy as np, pickle, json, time
from pathlib import Path

# 1) import パスの復旧
added_paths = []
for p in ["/work/scripts", "/work", "/work/vendor"]:
    if p not in sys.path:
        sys.path.insert(0, p)
        added_paths.append(p)
print(f"[PATH] inserted: {added_paths}")

# 2) new_model2 の import 検証
try:
    from new_model2.predict_model_v4_2_4 import full_walkforward
    from new_model2.feature_builder import WeatherFeatureBuilder
    print("[IMPORT] new_model2 OK: full_walkforward, WeatherFeatureBuilder")
except Exception as e:
    print(f"[IMPORT][ERROR] new_model2 読み込み失敗: {e}")
    raise

# 3) 365日データの用意（存在しない場合は df_all から作成）
if 'df_recent' not in globals() or df_recent is None or len(df_recent) == 0:
    assert 'df_all' in globals() and len(df_all) > 0, "df_all が未定義か空です"
    latest_date = pd.to_datetime(df_all["伝票日付"]).max().normalize()
    cutoff_date = latest_date - pd.Timedelta(days=365)
    mask = pd.to_datetime(df_all["伝票日付"]) >= cutoff_date
    df_recent = df_all.loc[mask].copy()
    print(f"[DATA] df_recent created: {df_recent['伝票日付'].min()} -> {df_recent['伝票日付'].max()} rows={len(df_recent)}")
else:
    print(f"[DATA] df_recent existing: {df_recent['伝票日付'].min()} -> {df_recent['伝票日付'].max()} rows={len(df_recent)}")

# 4) 祝日配列の確認（なければ補完）
if 'holidays' not in globals() or holidays is None or len(holidays) == 0:
    hol_min = pd.to_datetime(df_recent["伝票日付"]).min().normalize()
    hol_max = pd.to_datetime(df_recent["伝票日付"]).max().normalize()
    try:
        import jpholiday
        holidays = [d for d in pd.date_range(hol_min, hol_max) if jpholiday.is_holiday(d)]
        print(f"[HOLIDAYS] regen: {len(holidays)} days")
    except Exception as exc:
        holidays = []
        print(f"[HOLIDAYS][WARN] jpholiday 未使用: {exc} -> 空配列")
else:
    hol_min = pd.to_datetime(df_recent["伝票日付"]).min().normalize()
    hol_max = pd.to_datetime(df_recent["伝票日付"]).max().normalize()
    print(f"[HOLIDAYS] existing: {len(holidays)} from {hol_min.date()} to {hol_max.date()}")

# 5) 予約データを期間でフィルタ（なければスキップ）
if 'df_reserve' in globals() and df_reserve is not None and len(df_reserve) > 0:
    df_reserve_recent = df_reserve[(pd.to_datetime(df_reserve['予約日']) >= hol_min) & (pd.to_datetime(df_reserve['予約日']) <= hol_max)].copy()
    print(f"[RESERVE] rows={len(df_reserve_recent)} range={df_reserve_recent['予約日'].min()}->{df_reserve_recent['予約日'].max()}")
else:
    df_reserve_recent = pd.DataFrame(columns=["予約日"])
    print("[RESERVE][WARN] df_reserve が未定義 or 空 -> 空データで継続")

# 6) 天気特徴量の生成（失敗時はフォールバック）
try:
    wb = WeatherFeatureBuilder(hol_min, hol_max, enable_fallback=True)
    df_weather_full = wb.build()
    # index を DatetimeIndex(日単位)へ統一
    df_weather_full.index = pd.to_datetime(df_weather_full.index).tz_localize(None).floor("D")
    df_weather_full = df_weather_full.sort_index()
    print(f"[WEATHER] rows={len(df_weather_full)} cols={list(df_weather_full.columns)}")
except Exception as exc:
    print(f"[WEATHER][WARN] 生成失敗 -> フォールバック: {exc}")
    idx = pd.date_range(hol_min, hol_max, freq='D')
    df_weather_full = pd.DataFrame(index=idx)
    for c in ["天気_晴れ","天気_雨","天気_大雨","天気_台風"]:
        df_weather_full[c] = 0
    df_weather_full["天気_晴れ"] = 1
    print(f"[WEATHER] fallback rows={len(df_weather_full)}")

# 7) 特徴量リストの取得（ファイル優先, 失敗時は既存 current_keep）
feat_path = "/work/data/selected_features_final.txt"
current_keep_file = None
try:
    if Path(feat_path).exists():
        with open(feat_path, 'r', encoding='utf-8') as f:
            current_keep_file = [line.strip() for line in f if line.strip()]
        print(f"[FEATURES] loaded from file: {len(current_keep_file)} features")
except Exception as exc:
    print(f"[FEATURES][WARN] file read failed: {exc}")
if not current_keep_file:
    assert 'current_keep' in globals() and isinstance(current_keep, list) and len(current_keep) > 0, "current_keep が未定義 or 空"
    current_keep_file = current_keep
    print(f"[FEATURES] using notebook current_keep: { len(current_keep_file) } features")

# 8) Walk-forward（高速プロファイル）
MIN_STAGE1_DAYS_ = int(globals().get('MIN_STAGE1_DAYS', 14))
MIN_STAGE2_DAYS_ = int(globals().get('MIN_STAGE2_DAYS', 30))
TOP_N_ = int(globals().get('TOP_N', 5))
print(f"[CONFIG] MIN_STAGE1_DAYS={MIN_STAGE1_DAYS_} MIN_STAGE2_DAYS={MIN_STAGE2_DAYS_} TOP_N={TOP_N_}")
t0 = time.time()
actual, pred, last_model, dates = full_walkforward(
    df_recent, holidays, df_reserve_recent, df_weather_full,
    min_stage1_days=MIN_STAGE1_DAYS_, min_stage2_days=MIN_STAGE2_DAYS_,
    top_n=TOP_N_, allowed_features=current_keep_file, allowed_mode='whitelist',
    model_profile='fast', verbose=False,
 )
dt = time.time() - t0
print(f"[RUN] finished in {dt:.1f}s -> n={len(actual)}")

# 9) 評価と保存（結果＋メタ保存）
from sklearn.metrics import r2_score, mean_absolute_error
if len(actual) > 0:
    r2 = r2_score(actual, pred)
    mae = mean_absolute_error(actual, pred)
    print(f"[METRICS] R2={r2:.3f} MAE={mae:,.0f}kg n={len(actual)}")
else:
    r2 = float('nan'); mae = float('nan')
    print("[METRICS][WARN] 評価用データがありません")

out_path = "/work/data/final_stage1_model_predictable_365days_selected.pkl"
artifact = {
    "type": "walkforward_result",
    "module": "new_model2",
    "period": {"start": str(hol_min.date()), "end": str(hol_max.date())},
    "features": current_keep_file,
    "metrics": {"r2": float(r2), "mae": float(mae), "n": int(len(actual))},
    "dates": [str(pd.to_datetime(d).date()) for d in dates],
    "actual": list(map(float, actual)),
    "pred": list(map(float, pred)),
    "last_stage1_model": last_model,
}
with open(out_path, 'wb') as f:
    pickle.dump(artifact, f)
print(f"[SAVE] {out_path} saved (predict-ready: stage1 meta only)")

[PATH] inserted: ['/work/scripts', '/work', '/work/vendor']
[IMPORT] new_model2 OK: full_walkforward, WeatherFeatureBuilder
[DATA] df_recent existing: 2024-05-26 00:00:00 -> 2025-05-26 00:00:00 rows=57297
[HOLIDAYS] existing: 21 from 2024-05-26 to 2025-05-26
[RESERVE] rows=18856 range=2024-05-26 00:00:00->2025-05-26 00:00:00
[Weather] fetch 2024-05-26 -> 2025-05-26
[Weather] final params start_date=2024-05-26 end_date=2025-05-26


[INFO] 2025-09-12 10:54:54 new_model2.walkforward: ▶ full_walkforward(new_model2) start top_n=2 allowed_mode=whitelist
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [INIT] target_items=['混合廃棄物A', '混合廃棄物B']
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [INIT] target_items=['混合廃棄物A', '混合廃棄物B']


[Weather] status=200 url=https://archive-api.open-meteo.com/v1/archive?latitude=35.6895&longitude=139.6917&start_date=2024-05-26&end_date=2025-05-26&daily=temperature_2m_mean&daily=precipitation_sum&timezone=Asia%2FTokyo
[Weather] rows=366 cols=['平均気温', '降水量', '天気_大雨', '天気_晴れ', '天気_雨', '天気_台風']
[WEATHER] rows=366 cols=['平均気温', '降水量', '天気_大雨', '天気_晴れ', '天気_雨', '天気_台風']
[FEATURES] loaded from file: 15 features
[CONFIG] MIN_STAGE1_DAYS=30 MIN_STAGE2_DAYS=15 TOP_N=2


[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [FEATURES] use=15 (orig=23) mode=whitelist
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [CONFIG] dates=342 min_stage1_days=30 min_stage2_days=15 eff_stage2_rows=15
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [CONFIG] dates=342 min_stage1_days=30 min_stage2_days=15 eff_stage2_rows=15
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [DAY] 2024-07-05 (30/341) stage1_rows=0
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [DAY] 2024-07-05 (30/341) stage1_rows=0
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [DAY] 2024-07-06 (31/341) stage1_rows=1
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [DAY] 2024-07-06 (31/341) stage1_rows=1
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [DAY] 2024-07-07 (32/341) stage1_rows=2
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [DAY] 2024-07-07 (32/341) stage1_rows=2
[INFO] 2025-09-12 10:54:55 new_model2.walkforward: [DAY] 2024-07-08 (33/341) stage1_rows=3
[INFO] 2025-09-


===== ステージ1評価結果 =====
混合廃棄物A: R² = 0.756, MAE = 5,141kg
混合廃棄物B: R² = 0.449, MAE = 2,933kg
[RUN] finished in 65.8s -> n=297
[METRICS] R2=0.748 MAE=6,742kg n=297
[SAVE] /work/data/final_stage1_model_predictable_365days_selected.pkl saved (predict-ready: stage1 meta only)


In [28]:
# API用 Predictor を作成し、.predict() を持たせて保存
import pandas as pd, numpy as np, pickle, time
from typing import Union, Optional, Dict
from datetime import date, datetime
from pathlib import Path

# 前セルで import できている想定だが、念のため
from new_model2.feature_builder import WeightFeatureBuilder, ReserveFeatureBuilder
from new_model2.predict_model_v4_2_4 import train_and_predict_stage1
from sklearn.linear_model import ElasticNet

class NewModel2Predictor:
    """
    new_model2 のパイプラインを用いて、指定日のステージ1予測と合計（ステージ1の合算）を返す軽量予測器。
    注意: ここでは stage2（GBDT）による補正は行わず、ステージ1の合算を total として返します。
          将来的に stage2 も内包したい場合は、学習済み stage2 モデルの持ち回りか、
          直近日の履歴推論での簡易再学習を組み込みます。
    """
    def __init__(
        self, *,
        df_recent: pd.DataFrame,
        df_reserve_recent: pd.DataFrame,
        holidays: list,
        df_weather: Optional[pd.DataFrame],
        allowed_features: list,
        top_n: int = 2,
        min_stage1_days: int = 30,
    ):
        self.top_n = int(top_n)
        self.min_stage1_days = int(min_stage1_days)
        self.holidays = list(holidays) if holidays is not None else []
        self.allowed_features = [f for f in allowed_features if isinstance(f, str) and f]

        # 特徴量のベース（気象・予約は日別で後からマージ）
        self._df_raw = df_recent.copy()
        self._df_raw["伝票日付"] = pd.to_datetime(self._df_raw["伝票日付"]).dt.tz_localize(None).dt.floor("D")
        self._df_raw = self._df_raw.dropna(subset=["伝票日付"]).sort_values("伝票日付")

        wfb = WeightFeatureBuilder(self._df_raw, self._get_target_items(), self.holidays)
        self._df_feat_base, self._df_pivot = wfb.build()
        # 予約特徴量は全期間一括で算出
        try:
            self._df_reserve_all = ReserveFeatureBuilder(df_reserve_recent).build()
        except Exception:
            self._df_reserve_all = pd.DataFrame(index=pd.to_datetime([]))
        # 天気は渡されたものを保持（なければ空）
        self._df_weather_all = (df_weather.copy() if isinstance(df_weather, pd.DataFrame) else pd.DataFrame())
        # index 正規化
        for name, df in [("feat", self._df_feat_base),("pivot", self._df_pivot),("reserve", self._df_reserve_all),("weather", self._df_weather_all)]:
            try:
                if len(df) > 0:
                    df.index = pd.to_datetime(df.index).tz_localize(None).floor("D")
            except Exception:
                pass

    def _get_target_items(self):
        # df_recent の上位品目を決定
        return self._df_raw["品名"].value_counts().head(self.top_n).index.tolist()

    def _feature_list_for_today(self, df_today_columns):
        # allowed_features を today の列に存在するものでフィルタ
        return [c for c in self.allowed_features if c in df_today_columns] or [
            "合計_前日値","合計_前週平均","曜日","週番号","予約件数","予約合計台数"
        ]

    def predict(self, target_date: Union[str, date, datetime, pd.Timestamp]) -> Dict:
        tgt = pd.to_datetime(target_date).tz_localize(None).floor("D")
        if tgt not in self._df_feat_base.index:
            raise ValueError(f"target_date {tgt.date()} は学習期間のインデックスに存在しません。将来日の対応は別途実装が必要です。")

        # 過去ウィンドウ
        df_past_feat = self._df_feat_base[self._df_feat_base.index < tgt].tail(600)
        if len(df_past_feat) < self.min_stage1_days:
            raise ValueError(f"学習データ不足: {len(df_past_feat)} < min_stage1_days={self.min_stage1_days}")
        df_past_pivot = self._df_pivot.loc[df_past_feat.index]

        # 当日特徴量（後段で予約・天気をマージ）
        df_feat_today = self._df_feat_base.loc[[tgt]].copy()

        # 予約・天気の当日までを結合
        df_reserve_today = self._df_reserve_all[self._df_reserve_all.index <= tgt]
        df_weather_today = self._df_weather_all[self._df_weather_all.index <= tgt]

        def _merge(base: pd.DataFrame) -> pd.DataFrame:
            out = base.merge(df_reserve_today, left_index=True, right_index=True, how="left")
            out = out.merge(df_weather_today, left_index=True, right_index=True, how="left")
            return out.fillna(0)

        df_past_feat_m = _merge(df_past_feat)
        df_feat_today_m = _merge(df_feat_today)

        # 特徴量選択
        feat_list = self._feature_list_for_today(df_feat_today_m.columns)

        # ステージ1予測（メタ学習込み）
        stage1_eval = {item: {"y_true": [], "y_pred": []} for item in self._get_target_items()}
        stage1_result = train_and_predict_stage1(
            df_feat_today=df_feat_today_m,
            df_past_feat=df_past_feat_m,
            df_past_pivot=df_past_pivot,
            base_models=[('elastic', ElasticNet(alpha=0.1, l1_ratio=0.5))],
            meta_model_proto=ElasticNet(alpha=0.1, l1_ratio=0.5),
            feature_list=feat_list,
            target_items=self._get_target_items(),
            stage1_eval=stage1_eval,
            df_pivot=self._df_pivot,
        )
        # 出力整形
        per_item = {k.replace("_予測","" ): float(v) for k, v in stage1_result.items() if k.endswith("_予測")}
        total_pred = float(np.sum(list(per_item.values())))
        return {
            "date": str(tgt.date()),
            "per_item": per_item,
            "total": total_pred,
            "used_features": feat_list,
        }

    def predict_last(self) -> Dict:
        return self.predict(self._df_feat_base.index.max())

# インスタンス生成と保存
allowed = current_keep_file if 'current_keep_file' in globals() else current_keep
predictor = NewModel2Predictor(
    df_recent=df_recent,
    df_reserve_recent=df_reserve_recent,
    holidays=holidays,
    df_weather=df_weather_full,
    allowed_features=allowed,
    top_n=TOP_N_,
    min_stage1_days=MIN_STAGE1_DAYS_,
 )
api_model_path = "/work/data/final_stage1_model_api.pkl"
with open(api_model_path, "wb") as f:
    pickle.dump(predictor, f)
print(f"[SAVE][API] {api_model_path} saved")

# 簡易動作テスト
try:
    t0 = time.time()
    pred_sample = predictor.predict_last()
    print("[TEST] predict_last:", pred_sample)
    print(f"[TEST] elapsed {time.time()-t0:.2f}s")
except Exception as e:
    print(f"[TEST][WARN] predict_last failed: {e}")

[SAVE][API] /work/data/final_stage1_model_api.pkl saved
[TEST] predict_last: {'date': '2025-05-26', 'per_item': {'混合廃棄物A': 41500.37719924912, '混合廃棄物B': 10134.613893540725}, 'total': 51634.991092789845, 'used_features': ['曜日', '週番号', '祝日フラグ', '祝日前フラグ', '祝日後フラグ', '連休前フラグ', '連休後フラグ', '予約件数', '予約合計台数', '固定客予約数', '上位得意先予約数', '天気_晴れ', '天気_雨', '天気_大雨', '天気_台風']}
[TEST] elapsed 0.04s
